In [1]:
import os
import time
import pandas as pd
import yfinance as yf

CACHE_DIR = "market_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

SYMBOLS = {
    "EURUSD": "EURUSD=X",
    "GBPUSD": "GBPUSD=X",
    "USDJPY": "USDJPY=X",
    "AUDUSD": "AUDUSD=X",
    "USDCAD": "CAD=X",
    "USDCHF": "CHF=X",
    "NZDUSD": "NZDUSD=X",
    "XAUUSD": "GC=F",
    "BTCUSD": "BTC-USD"
}

TIMEFRAMES = {
    "M5":  ("5m",  "60d"),
    "M15": ("15m", "60d"),
    "H1":  ("1h",  "730d")
}


def resample_4h(df):

    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df.set_index("datetime")

    df4 = df.resample("4h").agg({
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum"
    })

    return df4.dropna().reset_index()


def download_market_data():

    for symbol, yf_symbol in SYMBOLS.items():

        print(f"\nDownloading {symbol} ({yf_symbol})")

        ticker = yf.Ticker(yf_symbol)

        for tf, (interval, period) in TIMEFRAMES.items():

            print(f"  → Fetching {tf}")

            try:

                df = ticker.history(period=period, interval=interval)

                if df.empty:
                    print("  ⚠ No data")
                    continue

                # Fix MultiIndex columns
                if isinstance(df.columns, pd.MultiIndex):
                    df.columns = df.columns.get_level_values(0)

                df = df.reset_index()

                time_col = "Datetime" if "Datetime" in df.columns else "Date"

                df = df.rename(columns={
                    time_col: "datetime",
                    "Open": "open",
                    "High": "high",
                    "Low": "low",
                    "Close": "close",
                    "Volume": "volume"
                })

                df = df[["datetime","open","high","low","close","volume"]]

                df["datetime"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_localize(None)

                df = df.drop_duplicates("datetime")
                df = df.sort_values("datetime")

                df = df.dropna(subset=["open","high","low","close"])

                if len(df) < 10:
                    print("  ⚠ Not enough data")
                    continue

                path = f"{CACHE_DIR}/{symbol}_{tf}.parquet"
                df.to_parquet(path)

                print(f"  ✅ Saved {len(df)} bars")

                if tf == "H1":

                    df4 = resample_4h(df)

                    path = f"{CACHE_DIR}/{symbol}_H4.parquet"
                    df4.to_parquet(path)

                    print(f"  ✅ Saved {len(df4)} H4 bars")

                time.sleep(1)

            except Exception as e:

                print(f"  ❌ Error: {e}")


download_market_data()


  → Fetching M5
  ✅ Saved 16491 bars
  → Fetching M15
  ✅ Saved 5505 bars
  → Fetching H1
  ✅ Saved 17266 bars
  ✅ Saved 4445 H4 bars

  → Fetching M5
  ✅ Saved 16498 bars
  → Fetching M15
  ✅ Saved 5508 bars
  → Fetching H1
  ✅ Saved 17267 bars
  ✅ Saved 4445 H4 bars

  → Fetching M5
  ✅ Saved 16192 bars
  → Fetching M15
  ✅ Saved 5412 bars
  → Fetching H1
  ✅ Saved 17153 bars
  ✅ Saved 4439 H4 bars

  → Fetching M5
  ✅ Saved 16836 bars
  → Fetching M15
  ✅ Saved 5622 bars
  → Fetching H1


$AUDUSD=X: possibly delisted; no price data found  (period=730d)


  ⚠ No data

  → Fetching M5
  ✅ Saved 16783 bars
  → Fetching M15
  ✅ Saved 5616 bars
  → Fetching H1
  ✅ Saved 17373 bars
  ✅ Saved 4460 H4 bars

  → Fetching M5
  ✅ Saved 16473 bars
  → Fetching M15
  ✅ Saved 5505 bars
  → Fetching H1
  ✅ Saved 17209 bars
  ✅ Saved 4446 H4 bars

  → Fetching M5
  ✅ Saved 16763 bars
  → Fetching M15
  ✅ Saved 5601 bars
  → Fetching H1
  ✅ Saved 17354 bars
  ✅ Saved 4459 H4 bars

  → Fetching M5
  ✅ Saved 13184 bars
  → Fetching M15
  ✅ Saved 4403 bars
  → Fetching H1
  ✅ Saved 13726 bars
  ✅ Saved 3720 H4 bars

  → Fetching M5
  ✅ Saved 17129 bars
  → Fetching M15
  ✅ Saved 5710 bars
  → Fetching H1
  ✅ Saved 17352 bars
  ✅ Saved 4341 H4 bars
